In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from retail_iq.preprocessing import load_raw_data, preprocess_dates, merge_datasets
from retail_iq.config import PLOT_DIR, OUTPUT_DIR

def run_cannibalization():
    print("Loading data for cannibalization analysis...")
    train, test, stores, oil, holidays, tx = load_raw_data()
    train, test, oil, holidays, tx = preprocess_dates([train, test, oil, holidays, tx])
    df = merge_datasets(train, stores, oil, holidays, tx)
    
    # 1. Cannibalization Detection
    def find_cannibal_pairs(df, promo_threshold=0, corr_threshold=-0.35):
        stores = df['store_nbr'].unique()
        pairs = []
        for s in stores:
            df_s = df[df['store_nbr'] == s].copy()
            # Residual sales (detrended via 28d rolling mean)
            df_s['sales_resid'] = df_s['sales'] - df_s.groupby('family')['sales'].transform(
                lambda x: x.shift(1).rolling(28, min_periods=1).mean()
            )
            promo_mask = df_s['onpromotion'] > promo_threshold
            df_promo = df_s[promo_mask].pivot_table(index='date', columns='family', values='sales_resid')
            corr = df_promo.corr()
            
            for i, fam_i in enumerate(corr.columns):
                for fam_j in corr.columns[i+1:]:
                    r = corr.loc[fam_i, fam_j]
                    if r < corr_threshold:
                        pairs.append({'store': s, 'family_i': fam_i, 'family_j': fam_j, 'r': r})
        return pd.DataFrame(pairs)
        
    cannibal_pairs = find_cannibal_pairs(df)
    
    # 2. Promotional Lift
    def compute_promo_lift(df, window_pre=28):
        df = df.sort_values(['store_nbr', 'family', 'date']).copy()
        df['baseline'] = df.groupby(['store_nbr', 'family'])['sales'].transform(
            lambda x: x.shift(1).rolling(window_pre, min_periods=1).mean()
        )
        # Avoid division by zero
        df['lift'] = (df['sales'] - df['baseline']) / df['baseline'].replace(0, np.nan)
        promo_events = df[df['onpromotion'] > 0][['store_nbr', 'family', 'date', 'sales', 'baseline', 'lift']].dropna()
        return promo_events
        
    promo_lift = compute_promo_lift(df)
    
    # Generate Report
    report_path = OUTPUT_DIR / 'reports'
    report_path.mkdir(parents=True, exist_ok=True)
    
    with open(report_path / 'cannibalization_report.md', 'w') as f:
        f.write("# Cannibalization and Promotional Lift Report\n\n")
        f.write("## Identified Cannibalization Pairs\n")
        if len(cannibal_pairs) > 0:
            f.write(cannibal_pairs.to_markdown(index=False) + "\n\n")
        else:
            f.write("No strong cannibalization pairs found (r < -0.35).\n\n")
            
        f.write("## Promotional Lift Analysis\n")
        if len(promo_lift) > 0:
            f.write(promo_lift.head(10).to_markdown(index=False) + "\n\n")
        else:
            f.write("No sufficient promotional lift events found.\n")
            
    # Optional Scatter plot
    if len(promo_lift) > 0:
        plt.figure(figsize=(10,6))
        # Group by promotion magnitude vs lift
        promo_data = df[df['onpromotion'] > 0]
        merged = promo_data.merge(promo_lift[['store_nbr', 'family', 'date', 'lift']], on=['store_nbr', 'family', 'date'])
        sns.scatterplot(x='onpromotion', y='lift', data=merged)
        plt.title('Promotional Lift vs OnPromotion Magnitude')
        plt.savefig(PLOT_DIR / 'promo_lift_scatter.png')
        plt.close()
        
    print("Cannibalization analysis complete.")

if __name__ == '__main__':
    run_cannibalization()


Loading data for cannibalization analysis...


Cannibalization analysis complete.
